# A/B/n inference: a practical textbook

This chapter follows an experiment from metric definitions to a launch decision.
We first compare two recommendation variants with control and then use historical
data to improve precision.

## Learning objectives

- map each metric type to a defensible null hypothesis;
- read an effect size, confidence interval and p-value together;
- distinguish family-wise error control from false-discovery control;
- derive CUPED and understand why it preserves the treatment-effect estimand;
- explain CUPAC cross-fitting and inspect variance-reduction diagnostics.

**Prerequisites:** Chapter 01 or equivalent knowledge of randomisation and power.
Every generated effect is known, so you can compare statistical output with the
data-generating truth.

## Chapter map and notation

1. Metrics and test selection.
2. Effect estimation and confidence intervals.
3. Multiple comparisons.
4. CUPED and CUPAC.
5. A decision checklist.

For arm $g$, $\mu_g=\mathbb E[Y_i\mid D_i=g]$ is the mean outcome. Our
usual estimand is $\tau_g=\mu_g-\mu_0$. A **p-value** is the probability,
under a specified null and test procedure, of a result at least as extreme
as the observed one. It is not $P(H_0\mid\text{data})$ and not the
probability that the result happened by chance alone.

> **Decision rule.** A confidence interval answers “which effect sizes are
> compatible with the data and model?” A launch rule should additionally
> include costs, risks and the MDE.

## 1. One experiment, several kinds of metrics

The data generator embeds revenue lifts of 8 and 15 for variants 1 and 2.
Conversion and acquisition channel are deliberately noisier, so the
corrected result will be more interesting than “everything is significant”.

In [1]:
import numpy as np
import pandas as pd

from hypex import ABTest
from hypex.dataset import (
    Dataset,
    FeatureRole,
    InfoRole,
    PreTargetRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils.tutorial_data_creation import DataGenerator

# The embedded lifts are known: variant 1 = 8, variant 2 = 15.
rng = np.random.default_rng(42)
n_users = 3_000
pre_revenue = rng.gamma(4, 25, n_users)
treat = rng.choice([0, 1, 2], n_users)
revenue = (
    pre_revenue
    + rng.normal(0, 20, n_users)
    + np.select([treat == 1, treat == 2], [8.0, 15.0])
)
conversion_probability = 1 / (1 + np.exp(-(-3 + 0.015 * revenue)))
converted = rng.binomial(1, conversion_probability)
channel = np.where(
    treat == 2,
    rng.choice(["organic", "paid"], n_users, p=[0.65, 0.35]),
    rng.choice(["organic", "paid"], n_users, p=[0.50, 0.50]),
)
ab_df = pd.DataFrame(
    {
        "user_id": np.arange(n_users),
        "treat": treat,
        "revenue": revenue,
        "converted": converted,
        "channel": channel,
    }
)
ab_df.head()

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


,user_id,treat,revenue,converted,channel
0,0,1,143.930845,0,paid
1,1,2,162.544979,0,organic
2,2,1,86.311347,0,organic
3,3,0,47.138858,0,paid
4,4,0,162.114694,1,organic


In [2]:
ab_data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(int),
        "revenue": TargetRole(float),
        "converted": TargetRole(int),
        "channel": TargetRole(str),
    },
    data=ab_df,
)

## 2. Choose a test for the question, not for the smallest p-value

| Metric / question | Useful test | What it checks |
|---|---|---|
| Continuous revenue, mean lift | Welch t-test | Difference in means without assuming equal variances |
| Skewed revenue, distributional shift | Mann–Whitney U | Stochastic ordering / distribution shift, not automatically a median test |
| Binary conversion | t-test on the indicator, or a proportion z-test in a custom pipeline | Difference in proportions |
| Categorical channel mix | Chi-square | Independence of channel and treatment |
| Entire numeric distribution | KS-test | Any distributional difference |

HypEx selects columns compatible with each test by role and dtype. A single
pipeline can therefore run all three tests without applying a numeric test
to `channel`.

### Welch's t-test: the mean comparison used below

For independent groups, Welch's statistic is

$$T=\frac{\bar Y_t-\bar Y_c}
{\sqrt{s_t^2/n_t+s_c^2/n_c}}.$$

Unlike Student's pooled test, it does not assume
$\sigma_t^2=\sigma_c^2$. Its degrees of freedom use the
Welch–Satterthwaite approximation:

$$\nu=\frac{(s_t^2/n_t+s_c^2/n_c)^2}
{(s_t^2/n_t)^2/(n_t-1)+(s_c^2/n_c)^2/(n_c-1)}.$$

The null is $H_0:\mu_t=\mu_c$. Rejecting it does not establish a
practically important effect, and failing to reject it does not establish
equivalence. Equivalence requires a margin and a dedicated analysis.

For a binary outcome $Y\in\{0,1\}$, $\bar Y=\hat p$, so the same mean
difference is a conversion-rate difference. A pooled proportion z-test
uses $\hat p=(x_t+x_c)/(n_t+n_c)$ under the null and

$$Z=\frac{\hat p_t-\hat p_c}
{\sqrt{\hat p(1-\hat p)(1/n_t+1/n_c)}}.$$

Mann–Whitney instead concerns
$P(Y_t>Y_c)+0.5P(Y_t=Y_c)$. It is a median test only under additional
shape assumptions. Chi-square compares observed cell counts $O_{ij}$ with
expected counts $E_{ij}=n_{i\cdot}n_{\cdot j}/n$:

$$\chi^2=\sum_{i,j}\frac{(O_{ij}-E_{ij})^2}{E_{ij}}.$$

Small expected counts invalidate the usual asymptotic approximation.

In [3]:
ab_result = ABTest(
    additional_tests=["t-test", "u-test", "chi2-test"],
    multitest_method="holm",
    equal_variance=False,
).execute(ab_data)
ab_result.summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value,UTest pass,UTest p-value,Chi2Test pass,Chi2Test p-value
0,revenue,1,99.345630,108.798293,9.452663,9.514926,4.829048,14.076279,OK,6.369077e-05,OK,5.688396e-06,NaN,NaN
1,revenue,2,99.345630,113.647569,14.301939,14.396143,9.543228,19.060651,OK,4.517482e-09,OK,5.365985e-10,NaN,NaN
2,converted,1,0.193806,0.216425,0.022619,11.670900,-0.012457,0.057694,NOT OK,2.063985e-01,NOT OK,2.066497e-01,NaN,NaN
3,converted,2,0.193806,0.230290,0.036484,18.825127,0.000327,0.072641,OK,4.810048e-02,OK,4.786663e-02,NaN,NaN
4,channel,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOT OK,0.491421
5,channel,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OK,0.000081


### From the table to a product statement

A defensible statement has four parts: **arm, metric, magnitude and
uncertainty**. For example: “Variant 1 changed mean revenue by the reported
`difference`, with the displayed 95% confidence interval.” Only then mention
the test result.

Under the same sampling assumptions, a two-sided 95% interval excluding
zero corresponds to $p<0.05$. The interval is more informative because it
also shows whether all compatible effects are too small to matter.

`difference %` is relative to the control mean and becomes unstable when
that mean is close to zero. It is not a percentage-point change: for
conversion, an absolute change of 0.02 is **2 percentage points**.

The confidence interval belongs to the mean difference. `OK` in this A/B
report means that the null was rejected; it is deliberately opposite to
the A/A convention. Statistical significance still does not tell us whether
the lift is economically worthwhile.

In [4]:
ab_result.sizes

,control size,test size,control size %,test size %,group
1,1001,1035,49,50,1
2,1001,964,50,49,2


## 3. Multiple comparisons: protect the decision family

We now have several metrics, variants and tests. If every raw p-value were
tested at 0.05 independently, the chance of at least one false discovery
would grow with the number of comparisons.

Holm is a step-down family-wise error-rate procedure. Order the p-values
$p_{(1)} \leq \dots \leq p_{(m)}$ and compare each with
$\alpha/(m-i+1)$, stopping at the first non-rejection. It is less
conservative than Bonferroni while controlling the same error rate under
arbitrary dependence.

The correction table is particularly useful here: a raw p-value just below
0.05 need not survive.

### FWER and FDR answer different questions

The family-wise error rate is
$\operatorname{FWER}=P(V\geq1)$, where $V$ is the number of false
rejections in the decision family. Bonferroni controls it by testing each
of $m$ hypotheses at $\alpha/m$.

The false-discovery rate is
$\operatorname{FDR}=\mathbb E[V/\max(R,1)]$, where $R$ is all rejections.
Benjamini–Hochberg is often appropriate for larger exploratory families,
but it answers a different question from Holm.

**A tiny Holm example.** For three raw p-values
$0.01,0.02,0.04$ and $\alpha=0.05$:

1. $0.01<0.05/3$, so reject the first;
2. $0.02<0.05/2$, so reject the second;
3. $0.04>0.05/1$, so stop and do not reject the third.

Adjusted p-values are the smallest family-wise level at which each
hypothesis would be rejected, with monotonicity enforced. They are not
recalibrated posterior probabilities.

In [5]:
ab_result.multitest

,field,test,old p-value,new p-value,correction,rejected,group
0,revenue,TTest,6.36907737695114e-05,0.00019107232130853423,0.3333333333333333,True,1
1,revenue,TTest,4.517482415358105e-09,1.806992966143242e-08,0.25,True,2
2,converted,TTest,0.20639845811772267,0.20639845811772267,1.0,False,1
3,converted,TTest,0.04810047767676886,0.09620095535353772,0.5,False,2
4,revenue,UTest,5.688395787221169e-06,1.7065187361663507e-05,0.33333333333333337,True,1
5,revenue,UTest,5.365985103562916e-10,2.1463940414251663e-09,0.25,True,2
6,converted,UTest,0.20664965825739878,0.20664965825739878,1.0,False,1
7,converted,UTest,0.04786662675677861,0.09573325351355721,0.5,False,2
8,channel,Chi2Test,0.4914206233979792,0.4914206233979792,1.0,False,1
9,channel,Chi2Test,8.087604235591947e-05,0.00016175208471183895,0.5,True,2


In [6]:
correction = ab_result.multitest.raw_data.copy()
old_pvalue = pd.to_numeric(correction["old p-value"])
rejected = correction["rejected"].map(
    lambda value: str(value).strip().lower() == "true"
)
correction["changed decision"] = (old_pvalue < 0.05) != rejected
correction.loc[
    correction["changed decision"],
    ["field", "test", "group", "old p-value", "new p-value", "rejected"],
]

,field,test,group,old p-value,new p-value,rejected
3,converted,TTest,2,0.04810047767676886,0.09620095535353772,False
7,converted,UTest,2,0.04786662675677861,0.09573325351355721,False


### Why the corrected decision changed

Find rows where the raw and adjusted decisions differ. These are not
contradictions: they are precisely the borderline discoveries for which
uncorrected analysis is too optimistic.

Also note which test enters the correction table. Running several tests on
one metric does not create independent replications of evidence; t- and
U-tests on the same rows are usually strongly correlated. Define the family
by the decisions those tests can influence, not by what happens to be
convenient after seeing the output.

Decide the comparison family **before** running the experiment. If revenue
is the only primary metric, correcting across every exploratory test may be
unnecessarily conservative. If all displayed tests drive the launch
decision, leaving them uncorrected is too optimistic. For false-discovery
rather than family-wise control, HypEx also exposes methods such as
`fdr_bh`.

## 4. Variance reduction: keep the estimand, remove avoidable noise

We switch to a two-arm dataset with two historical periods. The built-in
generator creates correlated covariates `X1`, `X2`, a lagged outcome and a
known treatment effect. Roles carry the temporal relationships:

- `PreTargetRole(parent="y", lag=...)`: historical outcomes;
- `FeatureRole(parent=..., lag=...)`: historical covariates;
- `TargetRole(cofounders=[...])`: features CUPAC may use.

No current post-treatment feature is used as a predictor.

In [7]:
generator = DataGenerator(
    n_samples=1_800,
    distributions={
        "X1": {"type": "normal", "mean": 1, "std": 2},
        "X2": {"type": "bernoulli", "p": 0.4},
        "y0": {"type": "normal", "mean": 60, "std": 12},
    },
    time_correlations={"X1": 0.7, "X2": 0.5, "y0": 0.8},
    effect_size=6.0,
    seed=7,
)
vr_df = generator.generate()
vr_df = vr_df.drop(columns=["y0", "z", "U", "D", "y1"]).rename(
    columns={"y0_lag_1": "y_lag1", "y0_lag_2": "y_lag2"}
)
vr_data = Dataset(
    roles={
        "d": TreatmentRole(),
        "y": TargetRole(cofounders=["X1", "X2"]),
        "y_lag1": PreTargetRole(parent="y", lag=1),
        "X1_lag1": FeatureRole(parent="X1", lag=1),
        "X2_lag1": FeatureRole(parent="X2", lag=1),
        "y_lag2": PreTargetRole(parent="y", lag=2),
        "X1_lag2": FeatureRole(parent="X1", lag=2),
        "X2_lag2": FeatureRole(parent="X2", lag=2),
    },
    data=vr_df,
    default_role=InfoRole(),
)
vr_df[["d", "y", "y_lag1", "y_lag2", "X1_lag1", "X2_lag1"]].head()

,d,y,y_lag1,y_lag2,X1_lag1,X2_lag1
0,1,91.858304,85.313739,84.988188,4.262584,0
1,1,53.624285,55.581030,77.444301,-3.246390,1
2,1,49.864548,52.267454,58.632841,-1.127689,0
3,0,50.897403,51.017097,44.792317,-1.138499,1
4,0,59.600132,59.261628,76.373987,0.132611,0


### CUPED in one equation

For a pre-experiment covariate $X$ centred at $\mathbb{E}X$,

$$
\widetilde{Y}=Y-\theta(X-\mathbb{E}X),
\qquad \theta=\frac{\operatorname{Cov}(Y,X)}{\operatorname{Var}(X)}.
$$

Centering preserves the expected mean under randomisation, while removing
the component predictable from history. In the ideal one-covariate case,
the variance reduction is approximately $\rho_{XY}^2$.

CUPAC generalises this idea: a model predicts $Y$ from historical data and
the prediction becomes the control variate. HypEx uses out-of-fold
predictions during model assessment to reduce target leakage.

### Derivation: why this $\theta$ minimises variance

Let $\widetilde Y=Y-\theta(X-\mathbb EX)$. For a constant $\theta$ and
pre-treatment $X$,

$$\operatorname{Var}(\widetilde Y)=
\operatorname{Var}(Y)+\theta^2\operatorname{Var}(X)
-2\theta\operatorname{Cov}(Y,X).$$

Differentiating with respect to $\theta$ and setting the derivative to zero
gives $\theta^*=\operatorname{Cov}(Y,X)/\operatorname{Var}(X)$.
Substitution yields

$$\operatorname{Var}(\widetilde Y)=
\operatorname{Var}(Y)(1-\rho_{XY}^2).$$

Under randomisation, $D\perp X$, so the adjustment has mean zero in each
arm and does not change the expected arm means. In finite samples the
estimated adjustment can move the point estimate, which is normal.

**When it can hurt.** A covariate measured after treatment, a covariate
uncorrelated with the outcome, leakage from the experiment, or fitting
noise can reduce or reverse the benefit. The percentage reported by HypEx
is descriptive of this sample, not a guarantee for the next experiment.

In [8]:
# Keep the unadjusted analysis as the transparent reference.
baseline_result = ABTest().execute(vr_data)
cuped_result = ABTest(cuped_features={"y": "y_lag1"}).execute(vr_data)
cupac_result = ABTest(
    enable_cupac=True,
    cupac_models=["linear", "ridge"],
).execute(vr_data)

### What the three runs change — and what they do not

`baseline_result` tests $Y$. `cuped_result` retains $Y$ and adds
$Y_{\text{cuped}}$ using one historical covariate. `cupac_result` retains
$Y$ and adds $Y_{\text{cupac}}$ using predictions from temporal features.

The treatment assignment and sample stay unchanged. We are comparing
**estimators of the same causal contrast**, not three treatments.

CUPAC model assessment follows a cross-fitting logic:

1. split historical observations into folds;
2. fit on some folds and predict the held-out fold;
3. collect out-of-fold predictions to assess variance reduction;
4. select the model and fit the adjustment for the experimental target.

Holding out folds prevents a flexible model from simply memorising the
outcome it is supposed to explain.

In [9]:
comparison_frames = []
for method, output in [
    ("raw", baseline_result),
    ("CUPED", cuped_result),
    ("CUPAC", cupac_result),
]:
    frame = output.summary.raw_data.copy()
    frame.insert(0, "run", method)
    comparison_frames.append(frame)

comparison = pd.concat(comparison_frames, ignore_index=True)
comparison["CI width"] = comparison["ci upper"] - comparison["ci lower"]
comparison[
    [
        "run",
        "feature",
        "difference",
        "ci lower",
        "ci upper",
        "CI width",
        "TTest p-value",
    ]
].round(4)

,run,feature,difference,ci lower,ci upper,CI width,TTest p-value
0,raw,y,9.3833,8.1273,10.6393,2.5121,0.0
1,CUPED,y,9.3833,8.1273,10.6393,2.5121,0.0
2,CUPED,y_cuped,8.7428,7.9318,9.5538,1.6220,0.0
3,CUPAC,y,9.3833,8.1273,10.6393,2.5121,0.0
4,CUPAC,y_cupac,8.6912,7.8837,9.4986,1.6149,0.0


### Reproduce the CUPED coefficient directly

The next cell uses only NumPy. It is a useful independent check of your
understanding: $\hat\theta$ is an ordinary least-squares slope of $Y$ on
the historical covariate $X$ with an intercept.

In [10]:
x = vr_df.y_lag1
y = vr_df.y
theta_manual = np.cov(y, x, ddof=1)[0, 1] / np.var(x, ddof=1)
adjusted_manual = y - theta_manual * (x - x.mean())
correlation = np.corrcoef(y, x)[0, 1]
manual_diagnostics = pd.DataFrame(
    {
        "theta": [theta_manual],
        "correlation": [correlation],
        "variance reduction, %": [100 * (1 - adjusted_manual.var() / y.var())],
        "1 - correlation squared": [1 - correlation**2],
    }
)
manual_diagnostics.round(4)

,theta,correlation,"variance reduction, %",1 - correlation squared
0,0.7988,0.7485,56.0204,0.4398


Compare this variance reduction with
`cuped_result.variance_reduction_report`. Small floating-point differences
are fine; a materially different value means the covariate mapping or the
data used in the two calculations is not the same.

Compare the adjusted target with the raw target **within each run**, not
the raw rows across different output objects. The point estimate can move
in a finite sample; the desired benefit is a shorter confidence interval,
not a larger effect or a smaller p-value by construction.

In [11]:
display(cuped_result.variance_reduction_report)
cupac_result.cupac.variance_reductions

,feature,variance_reduction_pct
0,y_cuped,56.020406


,target,best_model,variance_reduction_cv,variance_reduction_real
0,y,linear,64.020636,56.186777


In [12]:
cupac_result.cupac.feature_importances

,target,feature,importance,model
0,y,X1_lag2,0.137836,linear
1,y,X2_lag2,-0.169810,linear
2,y,y_lag2,0.793932,linear


### Translate variance reduction into sample efficiency

If variance falls by fraction $r=1-\operatorname{Var}(\widetilde Y)/
\operatorname{Var}(Y)$, the standard error at fixed sample size is
multiplied by $\sqrt{1-r}$. Equivalently, a raw estimator would need
approximately $1/(1-r)$ times as many observations for the same standard
error, under otherwise unchanged assumptions.

In [13]:
cuped_vr = (
    float(
        cuped_result.variance_reduction_report.raw_data["variance_reduction_pct"].iloc[
            0
        ]
    )
    / 100
)
cupac_vr = (
    float(
        cupac_result.cupac.variance_reductions.raw_data["variance_reduction_real"].iloc[
            0
        ]
    )
    / 100
)
efficiency = pd.DataFrame(
    {
        "method": ["CUPED", "CUPAC"],
        "variance reduction": [cuped_vr, cupac_vr],
        "SE multiplier": [np.sqrt(1 - cuped_vr), np.sqrt(1 - cupac_vr)],
        "raw-sample equivalent multiplier": [
            1 / (1 - cuped_vr),
            1 / (1 - cupac_vr),
        ],
    }
)
efficiency.round(3)

,method,variance reduction,SE multiplier,raw-sample equivalent multiplier
0,CUPED,0.560,0.663,2.274
1,CUPAC,0.562,0.662,2.282


This is an efficiency calculation, not permission to stop an experiment
early. Repeatedly peeking and stopping when an interval becomes narrow
changes the error properties of the procedure.

For CUPAC, `variance_reduction_cv` comes from historical cross-validation;
`variance_reduction_real` is measured on the adjusted experimental target.
Feature importances describe the selected predictive model. They are not
causal effects, and signed linear coefficients should not be read as
generic “importance” without considering scale.

## Decision checklist

- Predeclare primary metrics, the comparison family and the correction.
- Report effect sizes and intervals, not only rejections.
- Use only pre-treatment covariates for CUPED/CUPAC.
- Check that adjustment shortens intervals without introducing missingness
  or changing the target population.
- Keep a raw-metric analysis as a transparent reference.
- More variance reduction is useful only when the adjustment is valid;
  a high percentage alone is not proof of correctness.

**Next:** [Matching and observational causal inference](03_Matching_and_Causal_Inference.ipynb).

## 5. Final reading checklist

- [ ] The null hypothesis matches the metric and the product question.
- [ ] The reported effect has units, an interval and a business threshold.
- [ ] The comparison family was fixed before outcome analysis.
- [ ] Adjustment variables are pre-treatment and historically available.
- [ ] Variance reduction is assessed by precision, not by chasing p-values.
- [ ] Raw and adjusted analyses are both available for review.